# 2.3 Prompt evaluation: test datasets, running evals, grading

Step 2, lesson 3 of [Building with the Claude API](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287731) (Anthropic Academy).
Exam domains: **4 Prompt Engineering & Structured Output** (explicit criteria, iterating on prompts) and **5 Context Management & Reliability** (measuring before trusting an automated step).

What this notebook covers, in the course's order:

1. Why evaluate a prompt, and the eval workflow
2. Generating a test dataset with Claude
3. Running the prompt on every test case
4. Model-based grading: Claude scores the answers
5. Code-based grading: syntax checks
6. Putting it together: one score for the prompt

> **Model used here:** `claude-sonnet-5-5` for everything. The course generates its dataset with a faster model; `claude-haiku-5-5` would do for that step.
>
> **Cost:** 1 request for the dataset, then 2 per test case (answer + grade): 9 requests for 4 cases.

In [1]:
import ast
import json
import re
from pathlib import Path
from statistics import mean
from typing import Literal

from anthropic import Anthropic
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv()
client = Anthropic()
model = "claude-sonnet-5-5"

OUTPUTS = Path("outputs")
OUTPUTS.mkdir(exist_ok=True)


def text_of(message):
    """Return the text of the first text block in a response ("" if there is none)."""
    return next((block.text for block in message.content if block.type == "text"), "")


def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


def chat(messages, system=None, max_tokens=1000, stop_sequences=None, effort="low"):
    params = {"model": model, "max_tokens": max_tokens, "messages": messages, "output_config": {"effort": effort}}
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    return text_of(client.messages.create(**params))

## 1. Why evaluate, and the workflow

Trying a prompt on two or three inputs and shipping it is the common trap: the prompt then meets inputs you never tried. **Prompt evaluation** replaces "it looked fine" with a number you can compare between prompt versions:

1. **Draft** a prompt.
2. **Build a dataset** of realistic inputs (by hand, or generated by Claude).
3. **Run** the prompt on every input.
4. **Grade** every output, and average the grades.
5. **Change** the prompt, re-run, and keep the change only if the score goes up.

The task used here, as in the course: a prompt that solves small AWS-related coding requests, where each answer must be **Python**, **JSON** or a **regular expression**, with no explanation around it.

## 2. Generating a test dataset

Writing test cases by hand is slow, so the course asks Claude to write them. Each case has the task, the expected answer format, and **solution criteria**: what a good answer must do, so the grader judges against explicit points instead of a vague "is this good?".

> **Changed since the course.** The course got the dataset as JSON with prefill and a stop sequence. Here it comes through structured outputs (`client.messages.parse` with a Pydantic model, lesson 2.2), so it always parses.

In [2]:
class TestCase(BaseModel):
    task: str = Field(description="One short AWS-related coding task")
    format: Literal["python", "json", "regex"]
    solution_criteria: str = Field(description="What a correct solution must do, in one or two sentences")


class Dataset(BaseModel):
    test_cases: list[TestCase]


dataset_prompt = """Write 4 test cases for evaluating a prompt that solves small AWS-related coding tasks.

Each task must be solvable with ONE of these, and short:
- a single Python function
- a single JSON document (for example an IAM policy or an EventBridge pattern)
- a single regular expression

Use each format at least once. Keep every task small enough to answer in under 20 lines.
For each task, give solution criteria a reviewer can check."""

response = client.messages.parse(
    model=model,
    max_tokens=2000,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": dataset_prompt}],
    output_format=Dataset,
)
dataset = response.parsed_output.test_cases

(OUTPUTS / "03-eval-dataset.json").write_text(json.dumps([tc.model_dump() for tc in dataset], indent=2))
for i, tc in enumerate(dataset, 1):
    print(f"{i}. [{tc.format}] {tc.task}\n   criteria: {tc.solution_criteria}\n")

1. [python] Write a Python function named list_bucket_keys(bucket, prefix) that uses boto3 to return a list of all object keys in the given S3 bucket under the given prefix, handling pagination for buckets with more than 1000 objects.
   criteria: Uses boto3.client('s3') with a list_objects_v2 paginator (or a loop on ContinuationToken), collects every 'Key' from the 'Contents' entries while safely handling pages without 'Contents', and returns a list of strings.

2. [json] Write an IAM policy document in JSON that grants read-only access (s3:GetObject and s3:ListBucket) to the S3 bucket named 'my-data-bucket' and all objects in it, and nothing else.
   criteria: Valid JSON with Version '2012-10-17' and Allow statements; s3:ListBucket applies to arn:aws:s3:::my-data-bucket and s3:GetObject applies to arn:aws:s3:::my-data-bucket/*; no wildcard actions or other services are granted.

3. [regex] Write a regular expression that matches a valid AWS S3 bucket name between 3 and 63 characters 

## 3. Running the prompt on every test case

This is the prompt under evaluation, version 1. The answer has to be *only* code, so the course's prefill trick is replaced by the one from 2.2: ask for the answer inside `<solution>` tags and stop at the closing tag.

In [3]:
def run_prompt(test_case):
    """The prompt being evaluated (version 1). Returns the raw solution text."""
    prompt = f"""Please solve the following task:

{test_case.task}

* Respond only with Python, JSON, or a plain regular expression.
* Do not add comments, commentary or explanation.
* Put the solution inside <solution></solution> tags, without Markdown code fences."""
    text = chat([{"role": "user", "content": prompt}], max_tokens=1500, stop_sequences=["</solution>"])
    return text.split("<solution>", 1)[-1].strip()


first = run_prompt(dataset[0])
print(first)

import boto3


def list_bucket_keys(bucket, prefix):
    s3 = boto3.client("s3")
    paginator = s3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            keys.append(obj["Key"])
    return keys


## 4. Model-based grading

A **grader** turns an output into a score. The course names three kinds:

| Grader | Good for | Weak at |
|---|---|---|
| **Code** | exact, cheap checks: valid syntax, length, a keyword present | judging quality or meaning |
| **Model** (another Claude call) | quality, following instructions, completeness | cost; can be inconsistent without clear criteria |
| **Human** | anything, and checking the other graders | slow and expensive |

The model grader below gets the task, the solution and the **solution criteria**, and returns strengths, weaknesses, reasoning and a 1–10 score as structured output.

Two choices that make it more reliable:
- the score is the **last** field, so Claude writes its reasons *before* committing to a number;
- the prompt says what each score band means, instead of "rate this from 1 to 10".

In [4]:
class Grade(BaseModel):
    strengths: list[str]
    weaknesses: list[str]
    reasoning: str
    score: int = Field(description="1 to 10")


def grade_by_model(test_case, output):
    prompt = f"""You are an expert AWS engineer reviewing an AI-generated solution.

<task>
{test_case.task}
</task>

<solution>
{output}
</solution>

<criteria>
{test_case.solution_criteria}
</criteria>

Score the solution from 1 to 10:
- 10: meets every criterion, correct, nothing to fix
- 7-9: correct, with minor issues
- 4-6: works only partly, or misses a criterion
- 1-3: wrong, unusable, or not in the requested format
Be strict. List concrete strengths and weaknesses before deciding the score."""
    response = client.messages.parse(
        model=model,
        max_tokens=2000,
        output_config={"effort": "medium"},
        messages=[{"role": "user", "content": prompt}],
        output_format=Grade,
    )
    return response.parsed_output

## 5. Code-based grading

Some requirements are checkable with plain code, and then code is the better grader: exact, free and always the same. Here the check is "is the output valid in the format the task asked for?". Each validator returns 10 or 0. They run locally, so they can be tested without calling the API:

In [5]:
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(test_case, output):
    validator = {"json": validate_json, "python": validate_python, "regex": validate_regex}[test_case.format]
    return validator(output)


print(validate_json('{"Version": "2012-10-17"}'), validate_json("{'single': 'quotes'}"))
print(validate_python("def f(x):\n    return x"), validate_python("def f(x) return x"))
print(validate_regex(r"^i-[0-9a-f]{8,17}$"), validate_regex("[unclosed"))

10 0
10 0
10 0


## 6. Putting it together

`run_test_case` runs one case through the prompt and both graders; the final score of a case is the average of the model score and the syntax score. `run_eval` does that for the whole dataset and reports the average, which is the prompt's score.

In [6]:
def run_test_case(test_case, output=None):
    output = output if output is not None else run_prompt(test_case)
    grade = grade_by_model(test_case, output)
    syntax = grade_syntax(test_case, output)
    return {
        "task": test_case.task,
        "format": test_case.format,
        "output": output,
        "model_score": grade.score,
        "syntax_score": syntax,
        "score": (grade.score + syntax) / 2,
        "strengths": grade.strengths,
        "weaknesses": grade.weaknesses,
        "reasoning": grade.reasoning,
    }


def run_eval(dataset, first_output=None):
    results = []
    for i, test_case in enumerate(dataset):
        # reuse the answer already generated for case 0 above, to save one request
        output = first_output if (i == 0 and first_output is not None) else None
        results.append(run_test_case(test_case, output))
    return results


results = run_eval(dataset, first_output=first)
(OUTPUTS / "03-eval-results.json").write_text(json.dumps(results, indent=2))

print(f"{'format':<8}{'model':>6}{'syntax':>7}{'score':>7}  task")
for r in results:
    print(f"{r['format']:<8}{r['model_score']:>6}{r['syntax_score']:>7}{r['score']:>7.1f}  {r['task'][:70]}")
print(f"\nAverage score of prompt v1: {mean(r['score'] for r in results):.2f} / 10")

format   model syntax  score  task
python       9     10    9.5  Write a Python function named list_bucket_keys(bucket, prefix) that us
json        10     10   10.0  Write an IAM policy document in JSON that grants read-only access (s3:
regex        9     10    9.5  Write a regular expression that matches a valid AWS S3 bucket name bet
json        10     10   10.0  Write an EventBridge event pattern in JSON that matches EC2 instance s

Average score of prompt v1: 9.75 / 10


The number alone doesn't say what to fix; the grader's notes do. The weakest case:

In [7]:
worst = min(results, key=lambda r: r["score"])
print("Task:", worst["task"], "\n")
print(worst["output"], "\n")
print("Weaknesses:")
for w in worst["weaknesses"]:
    print(" -", w)
print("\nReasoning:", worst["reasoning"])

Task: Write a Python function named list_bucket_keys(bucket, prefix) that uses boto3 to return a list of all object keys in the given S3 bucket under the given prefix, handling pagination for buckets with more than 1000 objects. 

import boto3


def list_bucket_keys(bucket, prefix):
    s3 = boto3.client("s3")
    paginator = s3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            keys.append(obj["Key"])
    return keys 

Weaknesses:
 - No docstring or type hints
 - Creates a new S3 client on each call, which is slightly inefficient and makes testing harder (no injectable client)
 - No error handling for ClientError such as NoSuchBucket or AccessDenied, though letting them propagate is arguably acceptable

Reasoning: The solution meets every stated criterion: client, list_objects_v2 paginator, safe handling of missing Contents, and a returned list of keys. It is function

Two things worth noticing in this run:

- The grader marks the solution down for having **no docstring**, yet the prompt said "no comments". Unless the criteria say otherwise, a model grader judges whatever *it* thinks matters. When the prompt and the criteria disagree, align them (here: tell the grader that a bare solution, without comments, is what was asked for).
- Version 1 already scores close to 10. A score near the ceiling can't show whether a change helps; the fix is a harder dataset (edge cases, unusual inputs) on which the prompt sometimes fails.

An average can also hide a weak *kind* of input. Breaking the score down by format shows whether one of them is dragging (with 4 cases this is only an illustration; a real dataset needs many cases per group):

In [8]:
for fmt in ("python", "json", "regex"):
    scores = [r["score"] for r in results if r["format"] == fmt]
    if scores:
        print(f"{fmt:<7} {mean(scores):.1f} / 10 over {len(scores)} case(s)")

python  9.5 / 10 over 1 case(s)
json    10.0 / 10 over 2 case(s)
regex   9.5 / 10 over 1 case(s)


**Exam angle.**

- **Domain 4:** "explicit criteria before classifiers". The grader is consistent because each case carries written criteria and the score bands are defined. Vague instructions ("be strict", "check carefully") without criteria are the classic wrong answer.
- **Domain 4:** prefer a code check whenever the requirement is mechanical (valid JSON, a field present, a value in range); keep the model grader for judgement calls.
- **Domain 5:** an overall accuracy number can hide one category, document type or field that performs badly. Before trusting an automated step (or reducing human review), measure it **per segment** and keep sampling outputs for human checks.

## Recap

- Eval workflow: draft prompt → dataset → run → grade → change the prompt and compare scores.
- Claude can generate the dataset; give each case **solution criteria**.
- Model grader: structured output with reasons before the score and defined score bands.
- Code grader: exact checks (`json.loads`, `ast.parse`, `re.compile`) wherever possible.
- Report the average, read the weaknesses, and look at segments, not just the overall number.

Results are saved in `outputs/03-eval-results.json` (not committed).

Next: [2.4 Prompt engineering](04-prompt-engineering.ipynb), which uses this pipeline to measure each technique.